In [1]:
# ─────────────────────────────────────────────
# PART 1: Load pre-trained Word2Vec model
# ─────────────────────────────────────────────
!pip install gensim
import gensim.downloader as api

print("Loading Word2Vec model (Google News, ~1.6 GB)...")
model = api.load("word2vec-google-news-300")
print("Model loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.9 MB/s eta 0:00:00
Loading Word2Vec model (Google News, ~1.6 GB)...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Model loaded!


In [2]:
# ─────────────────────────────────────────────
# PART 2: Download analogy dataset
# ─────────────────────────────────────────────
import requests
from collections import defaultdict

url = "https://raw.githubusercontent.com/nicholas-leonard/word2vec/master/questions-words.txt"
response = requests.get(url)
lines = response.text.strip().split("\n")

# Parse into sections and word tuples
sections = defaultdict(list)
current_section = "general"

for line in lines:
    if line.startswith(":"):
        current_section = line[2:].strip()
    else:
        parts = line.lower().split()
        if len(parts) == 4:
            sections[current_section].append(tuple(parts))

print(f"Loaded {sum(len(v) for v in sections.values())} analogies across {len(sections)} categories")
for section, pairs in sections.items():
    print(f"  {section}: {len(pairs)} pairs")

Loaded 19544 analogies across 14 categories
  capital-common-countries: 506 pairs
  capital-world: 4524 pairs
  currency: 866 pairs
  city-in-state: 2467 pairs
  family: 506 pairs
  gram1-adjective-to-adverb: 992 pairs
  gram2-opposite: 812 pairs
  gram3-comparative: 1332 pairs
  gram4-superlative: 1122 pairs
  gram5-present-participle: 1056 pairs
  gram6-nationality-adjective: 1599 pairs
  gram7-past-tense: 1560 pairs
  gram8-plural: 1332 pairs
  gram9-plural-verbs: 870 pairs


In [3]:
# ─────────────────────────────────────────────
# PART 3: Manual analogy examples
# ─────────────────────────────────────────────

def word_analogy(w1, w2, w3, model, topn=5):
    """
    Compute w2 - w1 + w3 and find nearest word.
    Optimization: gensim's most_similar uses normalized vectors
    and NumPy matrix ops — no brute-force loop needed.
    Excludes input words from results automatically.
    """
    try:
        results = model.most_similar(positive=[w2, w3], negative=[w1], topn=topn)
        return results
    except KeyError as e:
        return f"Word not in vocabulary: {e}"

examples = [
    ("bad",   "good",      "excellent"),   # good - bad + excellent = ?
    ("late",  "delivery",  "fast"),        # delivery - late + fast = ?
    ("man",   "king",      "woman"),       # king - man + woman = queen (classic)
    ("paris", "france",    "berlin"),      # france - paris + berlin = germany
]

print("── Manual Analogy Examples ──\n")
for w1, w2, w3 in examples:
    result = word_analogy(w1, w2, w3, model)
    print(f"  {w2} - {w1} + {w3} =")
    if isinstance(result, list):
        for word, score in result:
            print(f"    → {word:20s}  (cosine similarity: {score:.4f})")
    else:
        print(f"    {result}")
    print()

── Manual Analogy Examples ──

  good - bad + excellent =
    → terrific              (cosine similarity: 0.6986)
    → superb                (cosine similarity: 0.6440)
    → fantastic             (cosine similarity: 0.6410)
    → great                 (cosine similarity: 0.6320)
    → exceptional           (cosine similarity: 0.5893)

  delivery - late + fast =
    → delievery             (cosine similarity: 0.4610)
    → twitch_muscles        (cosine similarity: 0.4286)
    → Delivery              (cosine similarity: 0.4249)
    → Eat##Hours.com        (cosine similarity: 0.3990)
    → www.BobBruss.com      (cosine similarity: 0.3976)

  king - man + woman =
    → queen                 (cosine similarity: 0.7118)
    → monarch               (cosine similarity: 0.6190)
    → princess              (cosine similarity: 0.5902)
    → crown_prince          (cosine similarity: 0.5499)
    → prince                (cosine similarity: 0.5377)

  france - paris + berlin =
    → germany        

In [4]:
# ─────────────────────────────────────────────
# PART 4: Evaluate on the dataset
# ─────────────────────────────────────────────

def evaluate_section(section_name, pairs, model, max_samples=100):
    correct = 0
    skipped = 0
    total = 0

    for w1, w2, w3, w4_expected in pairs[:max_samples]:
        # Skip if any word not in vocab
        if not all(w in model.key_to_index for w in [w1, w2, w3, w4_expected]):
            skipped += 1
            continue

        total += 1
        results = model.most_similar(positive=[w2, w3], negative=[w1], topn=1)
        predicted = results[0][0].lower()

        if predicted == w4_expected:
            correct += 1

    accuracy = correct / total if total > 0 else 0
    return correct, total, skipped, accuracy


print("── Evaluating on Dataset (100 samples per category) ──\n")
print(f"{'Category':<35} {'Correct':>8} {'Total':>8} {'Skipped':>8} {'Accuracy':>10}")
print("-" * 75)

overall_correct, overall_total = 0, 0

for section, pairs in sections.items():
    correct, total, skipped, acc = evaluate_section(section, pairs, model)
    overall_correct += correct
    overall_total += total
    print(f"{section:<35} {correct:>8} {total:>8} {skipped:>8} {acc:>9.1%}")

print("-" * 75)
overall_acc = overall_correct / overall_total if overall_total > 0 else 0
print(f"{'OVERALL':<35} {overall_correct:>8} {overall_total:>8} {'':>8} {overall_acc:>9.1%}")

── Evaluating on Dataset (100 samples per category) ──

Category                             Correct    Total  Skipped   Accuracy
---------------------------------------------------------------------------
capital-common-countries                  22       85       15     25.9%
capital-world                              4       31       69     12.9%
currency                                   2       21       79      9.5%
city-in-state                             21       98        2     21.4%
family                                    91      100        0     91.0%
gram1-adjective-to-adverb                 30      100        0     30.0%
gram2-opposite                            33      100        0     33.0%
gram3-comparative                         89      100        0     89.0%
gram4-superlative                         87      100        0     87.0%
gram5-present-participle                  72      100        0     72.0%
gram6-nationality-adjective                3       18       82  

In [5]:
# ─────────────────────────────────────────────
# PART 5: Show sample predictions from dataset
# ─────────────────────────────────────────────

print("── Sample Predictions from Dataset ──\n")

for section, pairs in list(sections.items())[:3]:   # show first 3 categories
    print(f"Category: {section}")
    shown = 0
    for w1, w2, w3, w4_expected in pairs:
        if not all(w in model.key_to_index for w in [w1, w2, w3]):
            continue
        results = model.most_similar(positive=[w2, w3], negative=[w1], topn=1)
        predicted = results[0][0].lower()
        status = "✓" if predicted == w4_expected else "✗"
        print(f"  {status}  {w2} - {w1} + {w3} = {predicted:15s}  (expected: {w4_expected})")
        shown += 1
        if shown == 5:
            break
    print()

── Sample Predictions from Dataset ──

Category: capital-common-countries
  ✗  greece - athens + baghdad = afghanistan      (expected: iraq)
  ✓  greece - athens + bangkok = thailand         (expected: thailand)
  ✗  greece - athens + beijing = europe           (expected: china)
  ✗  greece - athens + berlin = german           (expected: germany)
  ✓  greece - athens + cairo = egypt            (expected: egypt)

Category: capital-world
  ✗  nigeria - abuja + accra = madrid           (expected: ghana)
  ✗  nigeria - abuja + amman = mohammed         (expected: jordan)
  ✗  nigeria - abuja + ankara = buba             (expected: turkey)
  ✗  nigeria - abuja + athens = england          (expected: greece)
  ✗  nigeria - abuja + baghdad = iraqi            (expected: iraq)

Category: currency
  ✗  peso - argentina + brazil = pesos            (expected: real)
  ✗  peso - argentina + canada = philippine_peso  (expected: dollar)
  ✗  peso - argentina + croatia = colombian_peso   (expected: kuna)
